# donorvar on Parse — reproducing the corrected analysis with the package

Goal: check that the package gives the same result as the notebook code (section 13 of `aim1_compass_parse.ipynb`): in lymphocytes, typical donors differ little in response shape, while pairs involving interferon donors differ more.

Uses the cached pseudobulks (`data/parse10m/cache`), so nothing is recomputed from the raw file.

In [1]:
import os, numpy as np, pandas as pd
from itertools import combinations
import donorvar as dv

CACHE_DIR = "data/parse10m/cache"
RESULTS_DIR = "results/donorvar_parse"
os.makedirs(RESULTS_DIR, exist_ok=True)
INERT = ["Noggin", "Decorin", "PSPN", "GDNF", "Megalin", "PRL", "EPO", "TPO", "FGF-beta", "EGF", "HGF", "VEGF", "IGF-1"]
IFN_GROUP = ["Donor1", "Donor3", "Donor4", "Donor10"]
CELL_TYPES = ["B", "CD4 T", "CD8 T", "NK", "CD14 Mono", "CD16 Mono"]
N_GENES = 2000

# 1. wrap the existing cache as a donorvar table (keys: donor, cell_type, cytokine)
meta = pd.read_pickle(os.path.join(CACHE_DIR, "pb_meta.pkl"))[["donor", "cell_type", "cytokine", "half", "n_cells"]]
tab = dv.PseudobulkTable(np.load(os.path.join(CACHE_DIR, "pb_sums.npy")), meta,
                         np.load(os.path.join(CACHE_DIR, "pb_genes.npy"), allow_pickle=True))
print(tab.keys, tab.sums.shape)
GIDX = np.argsort(dv.lognorm(tab.sums.sum(0)))[::-1][:N_GENES]      # well-expressed genes

['donor', 'cell_type', 'cytokine'] (26162, 40352)


## 2. Well-noise energy per donor and cell type, from the replicate PBS wells

In [2]:
wsums = np.load(os.path.join(CACHE_DIR, "pbs_wells.npz"))["sums"]
wmeta = pd.read_pickle(os.path.join(CACHE_DIR, "pbs_wells_meta.pkl")).reset_index(drop=True)
WELL = {}
for (d, ct), g in wmeta.groupby(["donor", "cell_type"]):
    halves = {}
    for w, gw in g.groupby("well"):
        h = [gw[gw.half == k] for k in (0, 1)]
        if all(x.n_cells.sum() >= 20 for x in h):
            halves[w] = tuple(dv.lognorm(wsums[x.index.to_numpy()].sum(0).astype(float))[GIDX] for x in h)
    if len(halves) >= 3:
        WELL[(d, ct)] = dv.well_noise(halves)["well_energy"]
print(f"well energy for {len(WELL)} donor x cell type combinations")

well energy for 71 donor x cell type combinations


## 3. Corrected size/shape differences per responsive cytokine, then per pair type
Only cytokines with a clear response are used (energy above the 95th percentile of the inert cytokines): for cytokines without a response, estimated norms are near zero and the size part is dominated by noise.

In [3]:
donors = sorted(meta.donor.unique())
rows, KEPT = [], []
for ct in CELL_TYPES:
    conds = [c for c in meta.cytokine.unique() if c not in INERT + ["PBS"]]
    target = dv.common_target(tab, [(d, ct) for d in donors], conds + INERT)       # same depth for all donors
    resp = {d: dv.responses(tab, (d, ct), conds + INERT, reference=INERT, gene_idx=GIDX, target_cells=target)
            for d in donors}
    def halves_of(c): return {d: resp[d][c] for d in donors if c in resp[d]}
    def well_of(h): return {d: WELL.get((d, ct), 0.0) for d in h}
    # keep cytokines with a clear response: energy above the 95th percentile of the inert cytokines
    null = [dv.signal_energy(h, well_of(h)) for k in INERT if len(h := halves_of(k)) >= 6]
    if len(null) < 3: continue
    thr = np.quantile(null, 0.95)
    kept = {c: halves_of(c) for c in conds}
    kept = {c: h for c, h in kept.items() if len(h) >= 6 and dv.signal_energy(h, well_of(h)) > thr}
    # two normalizations: per cytokine (relative differences) and pooled over the cell type (one common scale)
    pooled = np.mean([dv.signal_energy(h, well_of(h)) for h in kept.values()]) if kept else None
    for c, h in kept.items():
        KEPT.append(dict(cell_type=ct, cytokine=c))
        for norm, en in [("per cytokine", None), ("pooled", pooled)]:
          for r in dv.size_shape(h, estimator="cross", well=well_of(h), energy=en):
            k = int(r["donor_d"] in IFN_GROUP) + int(r["donor_e"] in IFN_GROUP)
            rows.append(dict(normalization=norm, cell_type=ct, cytokine=c, pair=["non x non", "IFN x non", "IFN x IFN"][k], **r))
PAIRS = pd.DataFrame(rows)
PAIRS.to_csv(f"{RESULTS_DIR}/pairs.csv", index=False)
print("responsive cytokines kept:", pd.DataFrame(KEPT).groupby("cell_type").size().to_dict())
BY_PAIR = PAIRS.groupby(["normalization", "cell_type", "pair"])[["size", "shape"]].mean().round(3)
BY_PAIR.to_csv(f"{RESULTS_DIR}/by_pair.csv")
BY_PAIR

responsive cytokines kept: {'B': 40, 'CD14 Mono': 42, 'CD16 Mono': 45, 'CD4 T': 36, 'CD8 T': 33, 'NK': 32}


size  shape
normalization cell_type pair                   
per cytokine  B         IFN x IFN  0.835  0.451
                        IFN x non  0.882  0.544
                        non x non  0.668  0.146
              CD14 Mono IFN x IFN  0.690  0.414
                        IFN x non  0.711  0.499
                        non x non  0.735  0.250
              CD16 Mono IFN x IFN  0.511  0.432
                        IFN x non  0.758  0.467
                        non x non  0.922  0.221
              CD4 T     IFN x IFN  0.969  0.525
                        IFN x non  0.910  0.463
                        non x non  0.620  0.102
              CD8 T     IFN x IFN  0.994  0.529
                        IFN x non  0.894  0.471
                        non x non  0.485  0.112
              NK        IFN x IFN  0.833  0.526
                        IFN x non  0.768  0.532
                        non x non  0.479  0.191
pooled        B         IFN x IFN  0.440  0.371
                        IFN x non  0.620  0.600
                        non x non  0.380  0.151
              CD14 Mono IFN x IFN  0.173  0.358
                        IFN x non  0.275  0.542
                        non x non  0.242  0.300
              CD16 Mono IFN x IFN  0.176  0.371
                        IFN x non  0.352  0.491
                        non x non  0.324  0.249
              CD4 T     IFN x IFN  0.410  0.373
                        IFN x non  0.598  0.456
                        non x non  0.294  0.090
              CD8 T     IFN x IFN  0.402  0.347
                        IFN x non  0.587  0.452
                        non x non  0.251  0.118
              NK        IFN x IFN  0.309  0.330
                        IFN x non  0.488  0.517
                        non x non  0.251  0.175

**Compare with section 13 of the main notebook.** Values will not be identical (section 13 used only responsive cytokines and chose the thinning depth per cytokine), but the pattern should be the same: in B, CD4 T and CD8 T, `non x non` shape small and `IFN x non` shape larger.

Note: `common_target` takes the smallest group over all conditions, so rare cell types are thinned deeply; to keep more cells, pass fewer conditions or raise `min_cells_half`.